In [1]:
# Importar as bibliotecas para análise de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Carregar a base de dados do Perfect Store
df = pd.read_excel("../data/projeto1_vendas_perfect_store.xlsx")

# Visualizar as cinco primeiras linhas
df.head()

,mes_referencia,codigo_filial,cidade,regiao,score_disponibilidade_cardapio,score_padrao_visual,score_preco_correto,score_execucao_promocional,score_tempo_atendimento,faturamento_mensal,numero_pedidos,ticket_medio
0,2024-01-01,Filial 001,Rio de Janeiro,Sudeste,94.0,82.9,97.2,100.0,78.7,168293.15,3600,46.75
1,2024-01-01,Filial 002,São Paulo,Sudeste,91.3,100.0,87.4,68.5,74.0,133795.86,2202,60.76
2,2024-01-01,Filial 003,Porto Alegre,Sul,67.3,76.6,83.9,83.7,69.3,109384.81,1864,58.68
3,2024-01-01,Filial 004,Curitiba,Sul,80.4,94.0,100.0,85.1,83.7,154228.22,3077,50.12
4,2024-01-01,Filial 005,Curitiba,Sul,80.8,80.6,87.2,100.0,81.8,127114.45,2761,46.04


In [3]:
# Verificar a quantidade de linhas e colunas
print(f"Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas")

# Verificar os tipos de dados e valores não nulos
df.info()

Dimensões: 600 linhas e 12 colunas
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   mes_referencia                  600 non-null    datetime64[us]
 1   codigo_filial                   600 non-null    str           
 2   cidade                          600 non-null    str           
 3   regiao                          600 non-null    str           
 4   score_disponibilidade_cardapio  600 non-null    float64       
 5   score_padrao_visual             600 non-null    float64       
 6   score_preco_correto             600 non-null    float64       
 7   score_execucao_promocional      600 non-null    float64       
 8   score_tempo_atendimento         600 non-null    float64       
 9   faturamento_mensal              600 non-null    float64       
 10  numero_pedidos                  600 non-null    in

In [4]:
# Verificar registros duplicados
print(f"Registros duplicados: {df.duplicated().sum()}")

# Verificar a quantidade de filiais e meses
print(f"Filiais: {df['codigo_filial'].nunique()}")
print(f"Meses: {df['mes_referencia'].nunique()}")

# Verificar a quantidade de registros por filial
df.groupby("codigo_filial").size().describe()

Registros duplicados: 0
Filiais: 50
Meses: 12


count    50.0
mean     12.0
std       0.0
min      12.0
25%      12.0
50%      12.0
75%      12.0
max      12.0
dtype: float64

In [5]:
# Verificar duplicidade por filial e mês
print("Filial/mês duplicados:",
      df.duplicated(subset=["codigo_filial", "mes_referencia"]).sum())

# Selecionar as colunas de pontuação
score_cols = [col for col in df.columns if col.startswith("score_")]

# Verificar scores fora do intervalo de 0 a 100
print("Scores fora do intervalo:",
      ((df[score_cols] < 0) | (df[score_cols] > 100)).sum().sum())

Filial/mês duplicados: 0
Scores fora do intervalo: 0


## Cálculo do Perfect Store Score (PSS)

O PSS é calculado por meio de uma média ponderada de cinco indicadores operacionais, considerando a importância de cada critério para o desempenho das filiais.

In [6]:
# Definir os pesos dos indicadores operacionais
pesos = {
    "score_disponibilidade_cardapio": 0.30,
    "score_preco_correto": 0.25,
    "score_execucao_promocional": 0.20,
    "score_padrao_visual": 0.15,
    "score_tempo_atendimento": 0.10
}

# Calcular o Perfect Store Score de cada registro
df["pss"] = sum(df[coluna] * peso for coluna, peso in pesos.items())

# Visualizar os resultados
df[["codigo_filial", "mes_referencia", "pss"]].head()

,codigo_filial,mes_referencia,pss
0,Filial 001,2024-01-01,92.805
1,Filial 002,2024-01-01,85.340
2,Filial 003,2024-01-01,76.325
3,Filial 004,2024-01-01,88.610
4,Filial 005,2024-01-01,86.310


In [7]:
# Definir a classificação conforme o PSS
def classificar_pss(pss):
    if pss >= 90:
        return "Loja Perfeita"
    elif pss >= 75:
        return "Atenção"
    else:
        return "Crítico"

# Aplicar a classificação aos registros
df["classificacao_pss"] = df["pss"].apply(classificar_pss)

# Visualizar os resultados
df[["codigo_filial", "mes_referencia", "pss", "classificacao_pss"]].head()

,codigo_filial,mes_referencia,pss,classificacao_pss
0,Filial 001,2024-01-01,92.805,Loja Perfeita
1,Filial 002,2024-01-01,85.340,Atenção
2,Filial 003,2024-01-01,76.325,Atenção
3,Filial 004,2024-01-01,88.610,Atenção
4,Filial 005,2024-01-01,86.310,Atenção


In [8]:
# Contar os registros por classificação
distribuicao_pss = df["classificacao_pss"].value_counts()

# Calcular a distribuição percentual
percentual_pss = df["classificacao_pss"].value_counts(normalize=True) * 100

# Organizar os resultados em uma tabela
resumo_pss = pd.DataFrame({
    "Quantidade": distribuicao_pss,
    "Percentual (%)": percentual_pss.round(2)
})

# Exibir o resumo
resumo_pss

,Quantidade,Percentual (%)
classificacao_pss,,
Atenção,470,78.33
Loja Perfeita,111,18.50
Crítico,19,3.17


## Ranking anual das filiais

Consolidação dos indicadores mensais para avaliar o desempenho operacional e financeiro de cada filial durante 2024.

In [12]:
# Consolidar os indicadores anuais por filial
ranking_filiais = df.groupby(
    "codigo_filial",
    as_index=False
).agg(
    pss_medio=("pss", "mean"),
    faturamento_anual=("faturamento_mensal", "sum"),
    total_pedidos=("numero_pedidos", "sum")
)

# Ordenar as filiais pelo maior PSS médio
ranking_filiais = ranking_filiais.sort_values(
    by="pss_medio",
    ascending=False
).reset_index(drop=True)

# Verificar a quantidade de filiais no ranking
print(f"Total de filiais: {len(ranking_filiais)}")

# Visualizar as dez melhores filiais
ranking_filiais.head(10)

Total de filiais: 50


,codigo_filial,pss_medio,faturamento_anual,total_pedidos
0,Filial 045,88.858750,2240947.12,40942
1,Filial 050,88.217917,1957360.50,31480
2,Filial 004,88.008333,2253036.80,39885
3,Filial 001,87.870833,2327151.86,38938
4,Filial 017,87.478750,2269643.82,41251
5,Filial 011,87.226250,2287240.18,38544
6,Filial 027,87.135417,2029675.98,35885
7,Filial 009,87.124583,1711097.23,28980
8,Filial 032,86.984583,2102448.83,33357
9,Filial 040,86.980417,2235635.65,36721


In [10]:
# Verificar quantos grupos foram criados no ranking
print(f"Registros no ranking: {len(ranking_filiais)}")

# Verificar filiais associadas a mais de uma cidade
cidades_por_filial = df.groupby("codigo_filial")["cidade"].nunique()

# Exibir filiais com múltiplas cidades
print(cidades_por_filial[cidades_por_filial > 1])

Registros no ranking: 345
codigo_filial
Filial 001    7
Filial 002    7
Filial 003    8
Filial 004    8
Filial 005    6
Filial 006    7
Filial 007    8
Filial 008    6
Filial 009    7
Filial 010    7
Filial 011    6
Filial 012    8
Filial 013    6
Filial 014    6
Filial 015    6
Filial 016    7
Filial 017    9
Filial 018    8
Filial 019    7
Filial 020    7
Filial 021    7
Filial 022    6
Filial 023    6
Filial 024    8
Filial 025    8
Filial 026    7
Filial 027    8
Filial 028    8
Filial 029    5
Filial 030    6
Filial 031    5
Filial 032    9
Filial 033    7
Filial 034    8
Filial 035    6
Filial 036    8
Filial 037    7
Filial 038    8
Filial 039    6
Filial 040    6
Filial 041    8
Filial 042    6
Filial 043    6
Filial 044    7
Filial 045    6
Filial 046    5
Filial 047    7
Filial 048    6
Filial 049    7
Filial 050    7
Name: cidade, dtype: int64


In [11]:
# Verificar a quantidade de cidades e regiões por filial
localizacao_filiais = df.groupby("codigo_filial").agg(
    cidades_distintas=("cidade", "nunique"),
    regioes_distintas=("regiao", "nunique")
)

# Resumir as inconsistências cadastrais
print("Filiais com múltiplas cidades:",
      (localizacao_filiais["cidades_distintas"] > 1).sum())

print("Filiais com múltiplas regiões:",
      (localizacao_filiais["regioes_distintas"] > 1).sum())

# Visualizar os primeiros resultados
localizacao_filiais.head(10)

Filiais com múltiplas cidades: 50
Filiais com múltiplas regiões: 50


,cidades_distintas,regioes_distintas
codigo_filial,,
Filial 001,7,4
Filial 002,7,4
Filial 003,8,4
Filial 004,8,4
Filial 005,6,4
Filial 006,7,4
Filial 007,8,4
Filial 008,6,4
Filial 009,7,4


In [13]:
# Classificar as filiais pelo PSS médio anual
ranking_filiais["classificacao_anual"] = (
    ranking_filiais["pss_medio"].apply(classificar_pss)
)

# Contar as filiais por classificação anual
print(ranking_filiais["classificacao_anual"].value_counts())

# Visualizar as dez filiais com menor PSS médio
ranking_filiais.tail(10)

classificacao_anual
Atenção    50
Name: count, dtype: int64


,codigo_filial,pss_medio,faturamento_anual,total_pedidos,classificacao_anual
40,Filial 021,83.834583,2124821.03,36024,Atenção
41,Filial 029,83.693333,2130393.72,35464,Atenção
42,Filial 044,83.688333,2011586.44,34318,Atenção
43,Filial 048,83.461250,2276343.55,37953,Atenção
44,Filial 041,83.365417,2163784.68,35483,Atenção
45,Filial 010,82.922083,2272767.00,37545,Atenção
46,Filial 043,82.617083,2065485.52,36349,Atenção
47,Filial 003,82.536250,2172915.48,35135,Atenção
48,Filial 008,82.473333,2042145.87,35459,Atenção
49,Filial 005,82.454167,2274530.48,40388,Atenção


In [14]:
# Calcular a distância até a meta de excelência
ranking_filiais["pontos_para_meta"] = (
    90 - ranking_filiais["pss_medio"]
).clip(lower=0)

# Identificar as dez filiais mais próximas da meta
oportunidades = ranking_filiais.nsmallest(
    10, "pontos_para_meta"
)

# Visualizar as oportunidades de melhoria
oportunidades[
    ["codigo_filial", "pss_medio", "pontos_para_meta"]
]

,codigo_filial,pss_medio,pontos_para_meta
0,Filial 045,88.858750,1.141250
1,Filial 050,88.217917,1.782083
2,Filial 004,88.008333,1.991667
3,Filial 001,87.870833,2.129167
4,Filial 017,87.478750,2.521250
5,Filial 011,87.226250,2.773750
6,Filial 027,87.135417,2.864583
7,Filial 009,87.124583,2.875417
8,Filial 032,86.984583,3.015417
9,Filial 040,86.980417,3.019583
